In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [2]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [3]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [4]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')

In [5]:
# 数据预处理
df = pd.read_excel('../invertebrates_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [6]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)



smiles_endpoint_combined = [f"{sm}_{ep}" for sm, ep in zip(new_smiles_list, endpoints)]


groups = smiles_endpoint_combined  # 可直接用于 GroupKFold




In [7]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [8]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-16 07:59:16,108] A new study created in memory with name: no-name-52a6747f-f100-4e6c-9861-ebf8849b39b5
Training XGBoost: 100%|██████████| 10/10 [02:40<00:00, 16.01s/it]
[I 2025-05-16 08:01:56,224] Trial 0 finished with value: 0.7473767743951747 and parameters: {'n_estimators': 509, 'max_depth': 10, 'learning_rate': 0.04864408889292705, 'subsample': 0.8116073501688632, 'colsample_bytree': 0.6975484049668945, 'reg_alpha': 0.8853748738929744, 'reg_lambda': 0.10814255636439996}. Best is trial 0 with value: 0.7473767743951747.
Training XGBoost: 100%|██████████| 10/10 [01:05<00:00,  6.57s/it]
[I 2025-05-16 08:03:01,965] Trial 1 finished with value: 0.8723350839686906 and parameters: {'n_estimators': 325, 'max_depth': 5, 'learning_rate': 0.09688996041157205, 'subsample': 0.9560104555283371, 'colsample_bytree': 0.9312099659175446, 'reg_alpha': 0.4697920802899571, 'reg_lambda': 0.786901893181157}. Best is trial 0 with value: 0.7473767743951747.
Training XGBoost: 100%|██████████| 10/1

Best parameters for XGBoost: {'n_estimators': 532, 'max_depth': 17, 'learning_rate': 0.07197152412990382, 'subsample': 0.7611097576276309, 'colsample_bytree': 0.7814721400717639, 'reg_alpha': 0.5576788217747584, 'reg_lambda': 0.4071610613522757}
Best mean MAE: 0.6770


In [9]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)

[I 2025-05-16 09:57:25,718] A new study created in memory with name: no-name-2b9f56c3-d039-41c7-8966-934da0043cb7


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:40<00:00,  4.05s/it]
[I 2025-05-16 09:58:06,198] Trial 0 finished with value: 1.083903931857482 and parameters: {'n_estimators': 192, 'max_depth': 13, 'num_leaves': 281, 'learning_rate': 0.013219204341501523, 'feature_fraction': 0.9147574204018862, 'bagging_fraction': 0.9077984602574503, 'bagging_freq': 5, 'reg_alpha': 0.49643846460808894, 'reg_lambda': 0.6559641258507232}. Best is trial 0 with value: 1.083903931857482.
Training LightGBM: 100%|██████████| 10/10 [01:01<00:00,  6.17s/it]
[I 2025-05-16 09:59:07,972] Trial 1 finished with value: 0.9291388143123497 and parameters: {'n_estimators': 571, 'max_depth': 13, 'num_leaves': 154, 'learning_rate': 0.012160729383599897, 'feature_fraction': 0.7307890816359786, 'bagging_fraction': 0.6553178688644604, 'bagging_freq': 6, 'reg_alpha': 0.6445931979000534, 'reg_lambda': 0.335065723887096}. Best is trial 1 with value: 0.9291388143123497.
Training LightGBM: 100%|██████████| 10/10 [00:39<00:00,  3.9

Best parameters for LightGBM: {'n_estimators': 521, 'max_depth': 18, 'num_leaves': 176, 'learning_rate': 0.2375583608590192, 'feature_fraction': 0.8811490200434628, 'bagging_fraction': 0.9254197787783083, 'bagging_freq': 6, 'reg_alpha': 0.038765638556958096, 'reg_lambda': 0.5531625543351453}
Best mean MAE: 0.6917


In [11]:
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-16 11:46:31,363] A new study created in memory with name: no-name-17e56680-b46a-4fa6-9b72-ece860c3367f
[I 2025-05-16 11:50:07,881] Trial 0 finished with value: 0.4859066866563012 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'relu', 'alpha': 0.0014537924529504209, 'learning_rate_init': 0.00012050681821855591, 'solver': 'adam'}. Best is trial 0 with value: 0.4859066866563012.
[I 2025-05-16 11:53:52,843] Trial 1 finished with value: 1.5234136678131345 and parameters: {'hidden_layer_sizes': (150, 100, 50), 'activation': 'logistic', 'alpha': 0.0039059076267146243, 'learning_rate_init': 0.00032570372255465613, 'solver': 'sgd'}. Best is trial 0 with value: 0.4859066866563012.
[I 2025-05-16 11:57:34,897] Trial 2 finished with value: 0.8061266115008868 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'logistic', 'alpha': 5.4320423063983045e-05, 'learning_rate_init': 0.008818932796925076, 'solver': 'adam'}. Best is trial 0 with value: 0.4859066866563012.
[I


✅ Best Parameters Found:
{'hidden_layer_sizes': (100,), 'activation': 'tanh', 'alpha': 2.2009410033637966e-05, 'learning_rate_init': 0.00015486869614751136, 'solver': 'adam'}
Mean MAE = 0.4045
